<table>
  <tr>
    <td><div align="left"><font size="30">Two-tank system</font></div></td>
    <td><img src="https://raw.githubusercontent.com/petercorke/bdsim/main/docs/figs/bdsim_logo.png" width="300" height="120" style="height:auto"></td>
  </tr>
</table>

**Modeling the classic coupled two-tank system.**

(c) Peter Corke 2026

In [ ]:
# BDSIM_BOOTSTRAP_CELL: installs the package (Colab, JupyterLite or local), generated by sync_bootstrap.py from _bdsim_nb_bootstrap.py, do not edit

import subprocess
import sys
PACKAGE = 'bdsim'
IMPORT_NAME = 'bdsim'
GIT_URL: str | None = 'https://github.com/petercorke/bdsim'
JL_PACKAGES = []
JL_NO_DEPS: list[str] = []
JL_EXTRAS: dict[str, list[str]] = {'control': ['control']}
COLAB_PACKAGES = []

def _version_of(package: str, module) -> str:
    import importlib.metadata
    try:
        return importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        return str(getattr(module, '__version__', 'unknown'))

def _older(version: str, required: str) -> bool:
    try:
        from packaging.version import InvalidVersion, Version
        return Version(version) < Version(required)
    except (ImportError, InvalidVersion):
        return False

async def ensure_installed(extras: tuple[str, ...] | list[str]=(), packages: tuple[str, ...] | list[str]=(), min_version: str | None=None) -> bool:
    if sys.platform == 'emscripten':
        try:
            import piplite
            install = piplite.install
        except ImportError:
            import micropip
            install = micropip.install
        for name in JL_NO_DEPS:
            await install(name, deps=False)
        for name in JL_PACKAGES:
            await install(name)
        for extra in extras:
            for name in JL_EXTRAS.get(extra, []):
                await install(name)
        for name in packages:
            await install(name)
        await install(PACKAGE)
        where, colab = ('in browser', False)
    else:
        try:
            import google.colab
        except ImportError:
            where, colab = ('locally', False)
        else:
            print(f'Installing {PACKAGE}...')
            requirement = PACKAGE
            if extras:
                requirement += '[' + ','.join(extras) + ']'
            subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', requirement] + list(packages) + COLAB_PACKAGES, check=True)
            try:
                from matplotlib.backends import backend_registry
            except ImportError:
                pass
            else:
                clear = getattr(backend_registry, '_clear', None)
                if clear is not None:
                    clear()
            from google.colab import output
            output.enable_custom_widget_manager()
            where, colab = ('on Colab', True)
    import importlib
    module = importlib.import_module(IMPORT_NAME)
    version = _version_of(PACKAGE, module)
    if colab and min_version is not None and _older(version, min_version) and GIT_URL:
        print(f'{PACKAGE} {version} on PyPI is older than {min_version}, installing from GitHub...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--force-reinstall', '--no-deps', f'{PACKAGE} @ git+{GIT_URL}@main'], check=True)
        importlib.invalidate_caches()
        version = _version_of(PACKAGE, module)
        where += ', development version from GitHub'
    if not colab and where == 'locally':
        import importlib.metadata
        for name in packages:
            try:
                importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                print(f'This notebook needs {name}: pip install {name}')
    if min_version is not None and where in ('locally', 'on Colab') and _older(version, min_version):
        print(f'This notebook needs {PACKAGE} {min_version} or later: pip install -U {PACKAGE}')
    print(f'Running {where} using {PACKAGE} v{version}')
    return colab

COLAB = await ensure_installed(extras=['control'], min_version='1.5.0')


# Theory

The dynamics of the cart and pole system are taken from [Underactuated Robotics, Chapter 3, Russ Tedrake, MIT](https://underactuated.csail.mit.edu/acrobot.html#cart_pole).

The system comprises two tanks of water. A pump with controllable flow rate (system input) fillsthe first tank.
A manually adjustable valve connects the two tanks,
and another manually adjustable valve drains water from the second tank.  The aim is to control the height of 
water in the second tank.

The flow rate through a valve is given by
$$
Q = \gamma \sqrt{2 g \Delta h}
$$
where $\gamma = C_{d} A$,  $C_{d}$ is the discharge coefficient of the valve (typically between 0.6 and 0.8 for a fully open valve, accounting for turbulence and friction, and $A$ is the cross-sectional area of the valve opening, $g$ is gravitational acceleration, and $\Delta h$ is the difference in *head* across the valve.

The height of the two tanks can be written in terms of coupled first-order ODEs
$$
\begin{aligned}
\frac{d h_1}{dt} &= \frac{Q_i - Q_{12}}{A_1}\\
\frac{d h_2}{dt} &= \frac{Q_{12} - Q_{2}}{A_2}
\end{aligned}
$$
where the input flow rate is related to the control signal $0\le u \le 1$
$$
Q_i = Q_{max} u,
$$
the inter-tank flow rate is related to the height differential and $h_1 \ge h_2$ 
$$
Q_{12} = C_{d12} A \sqrt{2 g (h_1 - h_2)},
$$
and the tank 2 outflow is related to the water height of that tank
$$
Q_{2} = C_{d2} A \sqrt{2 g (h_1 - h_2)},\,\,.
$$

# Implementation with bdsim

We start with the standard imports, construct a simulator object, and use that to construct
a block diagram with all the `bdsim` blocks included as methods.

In [ ]:
import math
import numpy as np
import bdsim

sim = bdsim.BDSim(animation=True, animation_rate=10)  # create simulator

bd = sim.blockdiagram()  # create an empty block diagram

Next up, we define the parameters of the system

In [ ]:
r1= 0.3     # radius of first tank (m)
r2 = 0.2    # radius of second tank (m)
h = 1.0     # maximum height of tanks (m)
Qmax = 5e-3 # maximum flow rate (m^3/s)
g = 9.81    # gravity (m/s^2)
Cd = math.pi * 0.01**2 * 0.8
gamma_12 = Cd * 0.75
gamma_2 = Cd * 0.1

A1 = math.pi * r1**2  # area of first tank (m^2)
A2 = math.pi * r2**2  # area of second tank (m^2)

In [ ]:
A1*h / Qmax

The system can be modeled as two chains of integrators.  The first integrates cart acceleration $\ddot{x}$ to velocity $\dot{x}$ to position $x$.  The initial position and velocity is zero.

In [ ]:
h1 = bd.INTEGRATOR(name="h1")
h2 = bd.INTEGRATOR(name="h2")

In [ ]:
u = bd.STEP(T=5, on=0.8, off=0.0, name="u")  # input flow rate (m^3/s)
Qi = u * Qmax

In [ ]:
Q_12 = ((2*g*(h1 - h2)) >> bd.SQRT() ) * gamma_12  # flow rate from tank 1 to tank 2 (m^3/s)
Q_2 = ((2*g*h2) >> bd.SQRT() ) * gamma_2   # flow rate from tank 2 to environment (m^3/s)

In [ ]:
h1[0] = (Qi - Q_12) / A1   # dh1/dt = (Qi - Q_12) / A1
h2[0] = (Q_12 - Q_2) / A2  # dh2/dt = (Q_12 - Q_2) / A2

The system is initially in an unstable equilbrium with all states zero, the cart and pole not moving, and the pole pointing upwards.  We apply a force pulse ($t \in [1, 1.2]$) to the cart using the difference of two step functions

To include a graphical animation of our cart and pole system we will use simple Matplotlib 
patch graphics.  This kind of code is tedious to write, and can be outsourced to a coding agent.

In [ ]:
import matplotlib.patches as patches

WATER = "#74c0ee"  # light blue
OVERFLOW = "tab:red"  # a tank that has overflowed


def tanks_init(self, fig, ax, r1=r1, r2=r2, h=h):
    # The radii and height are bound when this is defined.  Tank 1 is drawn from x=0,
    # then a gap the width of a valve, then tank 2, all standing on y=0.
    v = 0.5 * min(r1, r2)  # width of a valve, and the gap between the tanks
    vh = 0.8 * v  # height of a valve
    left = [0, 2 * r1 + v]  # left side of each tank
    width = [2 * r1, 2 * r2]

    ax.set_xlim(-v, left[1] + width[1] + 3 * v)
    ax.set_ylim(-0.05 * h, 1.15 * h)
    ax.set_aspect("equal")
    ax.grid(True, linestyle="--")

    water = []
    for i in range(2):
        x0, x1 = left[i], left[i] + width[i]
        # the walls and floor, with no lid
        ax.plot([x0, x0, x1, x1], [h, 0, 0, h], color="black", lw=3, zorder=3)
        ax.text(x0 + width[i] / 2, 1.03 * h, f"tank {i + 1}", ha="center", va="bottom")
        # the water, with its bottom on the floor and no height until the first update
        rect = patches.Rectangle((x0, 0), width[i], 0, facecolor=WATER, edgecolor="none")
        ax.add_patch(rect)
        water.append(rect)

        # a valve at the lower right corner of the tank, a bowtie that is not animated
        cx, cy = x1 + v / 2, vh / 2
        for side in (-1, 1):
            tip = (cx, cy)
            top = (cx + side * v / 2, cy + vh / 2)
            bottom = (cx + side * v / 2, cy - vh / 2)
            ax.add_patch(patches.Polygon([tip, top, bottom], facecolor="black"))

    self.water = water
    self.h = h

and the actual update

In [ ]:
def tanks_update(self, t, u):
    for rect, level in zip(self.water, u):  # u is the level in each tank (m)
        # a level above the top of the tank is drawn full, and red
        rect.set_height(min(max(level, 0), self.h))
        rect.set_facecolor(OVERFLOW if level > self.h else WATER)

These two functions are passed to an `ANIMATION` block driven by the outputs of the `x` and `theta` blocks.  These inputs are the elements of `u` passed to `cartpole_update`.

In [ ]:

anim = bd.ANIMATION(
    init=tanks_init, update=tanks_update, nin=2, inputs=[h1, h2]
)


Our system is built, we will compile it, and display a summary

In [ ]:
bd.compile()  # check the diagram
print(bd)

Which has 47 blocks, although we have only explicitly created 4 integrators, 2 functions, 2 steps, an animation and a scope.  The majority of the blocks were created by the arithmetic expressions for the equations of motion and system energy.  

Here is a full list of blocks.

In [ ]:
# bd.report_summary()

where the implicitly created blocks have names prefixed by `_`.

To create all these blocks and connect them would be both tedious and error prone.  The power of `bdsim` is that it lets you move effortlessly between expression in terms of blocks or using Python expressions.

In [ ]:
out = sim.run(bd, T=60, watch=[h1, h2, Q_12, Q_2])  # simulate for 5s
print(out)


In [ ]:
lin1 = bd.linearize(t= 0, x=[0.2, 0.1],full=True)
lin2 = bd.linearize(t= 0, x=[0.5, 0.4],full=True)
print(lin1.poles)
print(lin2.poles)

Now we can design a linear controller using the Python Control package

```
$ pip install control
```

In [ ]:
import control as ct

# Create the state-space system (optional, but good practice)
sys = ct.ss(X.A, X.B, X.C, X.D, states=X.xindex.names, inputs=X.uindex.names, outputs=X.yindex.names)

# 2. Define the cost matrices
# Q must be positive semi-definite (penalizes state errors)
# R must be positive definite (penalizes control effort)
Q = np.diag([1, 1, 10, 1])  # weights for [x, x_dot, theta, theta_dot]
R = np.array([[100]]) # weight for control input (force F)

# 3. Compute the LQR optimal gain matrix K
# control.lqr returns:
# K: The optimal feedback gain matrix (u = -Kx)
# S: The solution to the algebraic Riccati equation
# E: The eigenvalues (poles) of the closed-loop system (A - BK)
K, S, E = ct.lqr(sys, Q, R)

# Note: You can also pass the arrays directly without creating a sys object:
# K, S, E = ct.lqr(A, B, Q, R)

print("Optimal Feedback Gain (K): \n", K)
print("Closed-loop Poles (E): \n", E)

In [ ]:
bd["_gain.0"].K = K  # set the gain block to the computed LQR gain
out = sim.run(bd, T=20, watch=[F, x, theta])  # simulate for 5s

In [ ]:
bd["_gain.0"].K=[1,2,3,4]
bd["_gain.0"].K